<a href="https://colab.research.google.com/github/KANISHKA-EEE/kanishka-codeboosters-2026/blob/main/Day_3/Day_3_data_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [58]:
!pip install requests --quiet

import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')

print('All libraries imported successfully!')
print(f'pandas : {pd.__version__}')
print(f'requests : {requests.__version__}')

All libraries imported successfully!
pandas : 2.2.2
requests : 2.32.4


In [59]:
raw_df=pd.read_csv('messy_sales_data.csv')
print(f'Raw data loaded:{raw_df.shape[0]} rows,{raw_df.shape[1]} columns')
print(f'Columns:{raw_df.columns.tolist()}')
print('\nFirst 5 rows:')
raw_df.head()

Raw data loaded:30 rows,9 columns
Columns:['order_id', 'customer_name', 'product', 'category', 'quantity', 'unit_price', 'order_date', 'city', 'sales_rep']

First 5 rows:


,order_id,customer_name,product,category,quantity,unit_price,order_date,city,sales_rep
0,1001,Ramesh Kumar,Laptop,Electronics,2.0,45000,2024-01-05,Mumbai,Anil Sharma
1,1002,Priya Nair,NaN,Electronics,1.0,15000,2024-01-07,Delhi,Sunita Rao
2,1003,AMIT VERMA,Keyboard,Accessories,3.0,1200,2024-01-08,Bangalore,Anil Sharma
3,1004,Sunita Patel,Monitor,Electronics,NaN,22000,2024-01-10,Chennai,Ravi Kumar
4,1005,Ramesh Kumar,Laptop,Electronics,2.0,45000,2024-01-05,Mumbai,Anil Sharma


In [60]:
#diagnose
print('DATA QUALITY DIAGONSIS REPORT')
#missing values
print(raw_df.isnull().sum())
#duplicates
print(f'\n[2] DUPLICATE ROWS:{raw_df.duplicated().sum()}')
#data types
print(raw_df.dtypes)
#unique values in text columns
print('\n[4] UNIQUE CATEGORIES:',raw_df['category'].unique)
print('[4] Sample customer names:',raw_df['customer_name'].dropna().unique()[:8])
print('[4] Sample order_date values:',raw_df['order_date'].unique()[:6])

DATA QUALITY DIAGONSIS REPORT
order_id         0
customer_name    2
product          1
category         1
quantity         3
unit_price       0
order_date       0
city             0
sales_rep        0
dtype: int64

[2] DUPLICATE ROWS:0
order_id           int64
customer_name     object
product           object
category          object
quantity         float64
unit_price         int64
order_date        object
city              object
sales_rep         object
dtype: object

[4] UNIQUE CATEGORIES: <bound method Series.unique of 0     Electronics
1     Electronics
2     Accessories
3     Electronics
4     Electronics
5     Accessories
6     Electronics
7     Accessories
8     Electronics
9     Accessories
10    Electronics
11    Accessories
12    Electronics
13    Electronics
14    Accessories
15    Accessories
16    Accessories
17    Electronics
18    Electronics
19    Accessories
20    Electronics
21    Accessories
22    Electronics
23    Electronics
24    Accessories
25    Electronics
26

In [61]:
#create a working copy
df=raw_df.copy()
print(f'Working copy created: {df.shape}')
print('raw_df is untouched - we can always reset by running df=raw_df.copy()')

Working copy created: (30, 9)
raw_df is untouched - we can always reset by running df=raw_df.copy()


In [62]:
df = raw_df.copy() # Reset df to ensure 'quantity' column is clean
print('Before fixing nulls:',df.isnull().sum().sum(),'total missing values')
df['customer_name'].fillna('Unknown Customer',inplace=True)
median_qty=df['quantity'].median()
df['quantity'].fillna(median_qty,inplace=True)
print(f'Filled missing quantity with median: {median_qty}')
df['category'].fillna('Uncategorized',inplace=True)
print('After fixing nulls:',df.isnull().sum().sum(),'total missing values')

Before fixing nulls: 7 total missing values
Filled missing quantity with median: 2.0
After fixing nulls: 1 total missing values


In [63]:
print("Missing Values in Product Column")
print(f'The missing values in product column count is:{df['product'].isnull().sum()}')
df['product'].fillna('Unknown Product',inplace=True)
print(f'After missing values handling:{df['product'].isnull().sum()}')

Missing Values in Product Column
The missing values in product column count is:1
After missing values handling:0


In [64]:
# Remove duplicates

print(f'Before deduplication: {len(df)} rows')
print(f'Duplicate rows: {df.duplicated().sum()}')

print('\nDuplicate rows:')
print(df[df.duplicated(keep=False)])

before_rows = len(df)

df.drop_duplicates(inplace=True)

print(f'After deduplication: {len(df)} rows')
print(f'Rows removed: {before_rows - len(df)}')

Before deduplication: 30 rows
Duplicate rows: 0

Duplicate rows:
Empty DataFrame
Columns: [order_id, customer_name, product, category, quantity, unit_price, order_date, city, sales_rep]
Index: []
After deduplication: 30 rows
Rows removed: 0


In [65]:
print('Sample dates before parsing:')
print(df['order_date'].head(8).tolist())
df['order_date']=pd.to_datetime(
    df['order_date'],
    dayfirst=True,
    errors='coerce'
)
nat_count=df['order_date'].isnull().sum()
print(f'\nUnparsable dates(NaT):{nat_count}')
df['year']=df['order_date'].dt.year
df['month']=df['order_date'].dt.month
df['month_name']=df['order_date'].dt.strftime('%B')
print('\nSample dates after parsing:')
print(df[['order_date','year','month','month_name']].head(5))

Sample dates before parsing:
['2024-01-05', '2024-01-07', '2024-01-08', '2024-01-10', '2024-01-05', '07-01-2024', '2024-01-12', '2024-01-13']

Unparsable dates(NaT):17

Sample dates after parsing:
  order_date    year  month month_name
0 2024-05-01  2024.0    5.0        May
1 2024-07-01  2024.0    7.0       July
2 2024-08-01  2024.0    8.0     August
3 2024-10-01  2024.0   10.0    October
4 2024-05-01  2024.0    5.0        May


In [66]:
print('Sample dates before parsing:')
print(df['order_date'].head(8).tolist())
df['order_date']=pd.to_datetime(
    df['order_date'],
    dayfirst=True,
    errors='coerce'
)
nat_count=df['order_date'].isnull().sum()
print(f'\nUnparsable dates(NaT):{nat_count}')
df['year']=df['order_date'].dt.year
df['month']=df['order_date'].dt.month
df['month_name']=df['order_date'].dt.strftime('%B')
df['order_date'] = df['order_date'].dt.strftime('%d-%m-%Y')
print('\nSample dates after parsing:')
print(df[['order_date','year','month','month_name']].head(5))

Sample dates before parsing:
[Timestamp('2024-05-01 00:00:00'), Timestamp('2024-07-01 00:00:00'), Timestamp('2024-08-01 00:00:00'), Timestamp('2024-10-01 00:00:00'), Timestamp('2024-05-01 00:00:00'), NaT, Timestamp('2024-12-01 00:00:00'), NaT]

Unparsable dates(NaT):17

Sample dates after parsing:
   order_date    year  month month_name
0  01-05-2024  2024.0    5.0        May
1  01-07-2024  2024.0    7.0       July
2  01-08-2024  2024.0    8.0     August
3  01-10-2024  2024.0   10.0    October
4  01-05-2024  2024.0    5.0        May


In [67]:
print('Sample dates before parsing:')
print(df['order_date'].head(8).tolist())
df['order_date']=pd.to_datetime(
    df['order_date'],
    dayfirst=True,
    errors='coerce'
)
nat_count=df['order_date'].isnull().sum()
print(f'\nUnparsable dates(NaT):{nat_count}')
df['year']=df['order_date'].dt.year
df['month']=df['order_date'].dt.month
df['month_name']=df['order_date'].dt.strftime('%B')
df['day_name']=df['order_date'].dt.day_name()
df['order_date'] = df['order_date'].dt.strftime('%d-%m-%Y')
print('\nSample dates after parsing:')
print(df[['order_date','year','month','month_name','day_name']].head(5))

Sample dates before parsing:
['01-05-2024', '01-07-2024', '01-08-2024', '01-10-2024', '01-05-2024', nan, '01-12-2024', nan]

Unparsable dates(NaT):17

Sample dates after parsing:
   order_date    year  month month_name   day_name
0  01-05-2024  2024.0    5.0        May  Wednesday
1  01-07-2024  2024.0    7.0       July     Monday
2  01-08-2024  2024.0    8.0     August   Thursday
3  01-10-2024  2024.0   10.0    October    Tuesday
4  01-05-2024  2024.0    5.0        May  Wednesday
